# Stateful Chatbot

In [ ]:
from IPython.display import Markdown, display, update_display
from openai import OpenAI

### GPT setup

In [ ]:
# import os
# from dotenv import load_dotenv

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if not api_key:
#     print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# elif not api_key.startswith("sk-proj-"):
#     print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# elif api_key.strip() != api_key:
#     print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
# else:
#     print("API key found and looks good so far!")

# MODEL = "gpt-5-nano"
# 
# my_ai = OpenAI()

### Ollama setup

In [ ]:
BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"
MODEL = "llama3.2"

my_ai = OpenAI(base_url=BASE_URL, api_key=API_KEY)

#### Other setup

In [ ]:
# MODEL = "deepseek-r1:1.5b"

### Memorable prompts

In [ ]:
# SYSTEM_PROMPT = (
#     "You are a helpless assistant that doesn't know anything. "
#     "You make up answers as you go along because you are afraid of being judged. "
#     "You answer with absolute confidence and a serious tone, but most of the information you provide is wrong or made up. "
#     "You are having a conversation with me so your answers shouldn't be too long. "
#     "Reply in markdown. Do not wrap the markdown in a code block - respond just with the markdown."
# )

# THINKING_PROMPT = "First describe your thought process then give the answer"

## System/user prompts

### Templates

In [ ]:
MARKDOWN_PROMPT = "Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."

### System

In [ ]:
SYSTEM_PROMPT = (
    "You are a basic assistant. "
    "You solve simple tasks. "
    "Your answers are serious and straight to the point. "
    + MARKDOWN_PROMPT
)

## Helpers

In [ ]:
def display_stream(stream):
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response

In [ ]:
def do_chat(user_prompt, messages):
    print("Q:", user_prompt)

    print("Answering...")

    response = my_ai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )
    full_response = display_stream(response)
    messages.append({"role": "assistant", "content": full_response})

    user_prompt = input("Wanna go further? Enter your prompt, otherwise just press enter: ")

    if user_prompt:
        messages.append({"role": "user", "content": user_prompt})
        print("Answering...")
        do_chat(user_prompt, messages)

## Actual Chat

In [ ]:
user_prompt = input("Enter your first prompt: ")
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt}
]
do_chat(user_prompt, messages)

### Debug

In [ ]:
headers = messages[0].keys()

table = "| " + " | ".join(headers) + " |\n"
table += "| " + " | ".join("---" for _ in headers) + " |\n"

for row in messages:
    table += "| " + " | ".join(str(row.get(h, "")) for h in headers) + " |\n"

display(Markdown(table))